# 📦 Store Item Demand Forecasting — Standalone End-to-End Pipeline

A **fully self-contained** version of the production pipeline: every function and class
below is a verbatim copy of the code in `src/` and `main.py`, so this notebook runs
**without the project scripts** and — thanks to fixed random seeds throughout —
reproduces **exactly the same numbers, figures, and comparison table** as `python main.py`.

**Only requirements:** the Python packages in `requirements.txt` and an internet
connection for the first-time dataset download (913,000 rows, ~17 MB, public mirror —
no Kaggle credentials needed).

> ⏱️ **Runtime note:** the training stage tunes three gradient-boosting models on
> 821k rows — expect **~15–20 minutes** end to end on a modern machine.

**Stages:** load → clean → EDA → feature engineering → chronological split →
feature-importance analysis → preprocessing → training (LightGBM, XGBoost,
HistGradientBoosting, tuned with time-series CV) → evaluation & comparison (SMAPE) →
SHAP explainability.

In [ ]:
# Imports — everything the pipeline needs, nothing project-specific
import time
import logging
import warnings
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, make_scorer,
)
import xgboost as xgb
import lightgbm as lgb
import shap

%matplotlib inline

# Same warning filters as main.py
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# The pipeline modules log through `logger`; a plain stdlib logger reproduces
# the same messages as notebook output
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s | %(message)s", force=True)
logger = logging.getLogger("demand-notebook")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

## ⚙️ Configuration

The exact constants from `src/config.py`: seed, chronological split boundaries,
feature lists, and the hyperparameter search spaces. Paths anchor to the project root
whether the notebook is launched from `notebooks/` or the repository root.

In [ ]:
# Paths — anchored to the project root
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

RAW_DATA_FILE = PROJECT_ROOT / "data" / "raw" / "store-item-demand-train.csv"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
for _dir in [RAW_DATA_FILE.parent, FIGURES_DIR, REPORTS_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)

# Dataset source (public GitHub mirror — no Kaggle credentials required)
DATASET_URL = 'https://raw.githubusercontent.com/DharitShah13/Kaggle-Store-Item-Demand-Forecasting-Challenge/master/train.csv'

# Reproducibility
RANDOM_STATE = 42

# Chronological split boundaries — random splitting would leak the future
# into training. Val/test windows are 3 months, matching the competition's
# forecast horizon.
TRAIN_END_DATE = '2017-06-30'
VAL_END_DATE = '2017-09-30'

# Target / identifiers
TARGET = 'sales'
DATE_COLUMN = 'date'
N_STORES = 10
N_ITEMS = 50

# Feature definitions
RAW_FEATURES = ['date', 'store', 'item']
CALENDAR_NUMERIC = ['year',
 'day',
 'dayofyear',
 'weekofyear',
 'month_sin',
 'month_cos',
 'dow_sin',
 'dow_cos',
 'doy_sin',
 'doy_cos']
AGGREGATE_FEATURES = ['store_item_mean', 'store_item_dow_mean', 'item_month_mean', 'store_month_mean']
CALENDAR_CATEGORICAL = ['month', 'dayofweek']
FLAG_FEATURES = ['is_weekend', 'is_month_start', 'is_month_end']
ENTITY_FEATURES = ['store', 'item']
NUMERIC_FEATURES = ENTITY_FEATURES + CALENDAR_NUMERIC + AGGREGATE_FEATURES

# Hyperparameter search spaces
LIGHTGBM_PARAMS = {'regressor__n_estimators': [300, 500, 800],
 'regressor__learning_rate': [0.03, 0.05, 0.1],
 'regressor__num_leaves': [31, 64, 128],
 'regressor__max_depth': [-1, 8, 12],
 'regressor__subsample': [0.8, 1.0],
 'regressor__colsample_bytree': [0.8, 1.0],
 'regressor__min_child_samples': [20, 50, 100],
 'regressor__reg_lambda': [0.0, 0.1, 1.0]}

XGBOOST_PARAMS = {'regressor__n_estimators': [300, 500, 800],
 'regressor__learning_rate': [0.03, 0.05, 0.1],
 'regressor__max_depth': [6, 8, 10],
 'regressor__subsample': [0.8, 1.0],
 'regressor__colsample_bytree': [0.8, 1.0],
 'regressor__min_child_weight': [1, 5, 10],
 'regressor__reg_lambda': [0.5, 1.0, 2.0]}

HISTGB_PARAMS = {'regressor__max_iter': [300, 500, 800],
 'regressor__learning_rate': [0.03, 0.05, 0.1],
 'regressor__max_leaf_nodes': [31, 63, 127],
 'regressor__min_samples_leaf': [20, 50, 100],
 'regressor__l2_regularization': [0.0, 0.1, 1.0]}

# Training configuration — TimeSeriesSplit folds inside the random search
CV_FOLDS = 2
N_ITER_SEARCH = 8

np.random.seed(RANDOM_STATE)   # same global seed as main.py

## 1️⃣ Data Loading & Validation

Downloads the dataset on first use (10 stores × 50 items × 5 years = 500 complete
daily series, 913,000 rows), verifies the expected schema, and profiles coverage,
duplicates, and target statistics.

In [ ]:
# Expected columns in the raw dataset — used as a sanity check
EXPECTED_COLUMNS = ['date', 'store', 'item', 'sales']


def download_dataset(url: str = DATASET_URL, dest: Path = RAW_DATA_FILE) -> Path:
    """
    Download the dataset from a direct URL if not already present.

    Args:
        url: Direct download URL for the CSV file.
        dest: Local file path to save the downloaded CSV.

    Returns:
        Path to the downloaded (or existing) file.

    Raises:
        requests.HTTPError: If the download fails.
    """
    if dest.exists():
        logger.info(f"Dataset already exists at {dest} — skipping download.")
        return dest

    logger.info(f"Downloading dataset from {url}...")
    dest.parent.mkdir(parents=True, exist_ok=True)

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    dest.write_bytes(response.content)
    logger.info(f"Dataset saved to {dest} ({dest.stat().st_size / 1024**2:.1f} MB)")
    return dest

def load_raw_data(filepath: Path = RAW_DATA_FILE) -> pd.DataFrame:
    """
    Load the raw CSV dataset with type parsing.

    Args:
        filepath: Path to the raw CSV file.

    Returns:
        DataFrame with a parsed datetime `date` column.

    Raises:
        FileNotFoundError: If the file doesn't exist.
        ValueError: If the schema doesn't match expectations.
    """
    if not filepath.exists():
        raise FileNotFoundError(
            f"Dataset not found at {filepath}. Run download_dataset() first."
        )

    logger.info(f"Loading dataset from {filepath}...")
    df = pd.read_csv(filepath, parse_dates=[DATE_COLUMN])

    # --- Schema validation ---
    missing_cols = set(EXPECTED_COLUMNS) - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing expected columns: {missing_cols}")

    logger.info(
        f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns "
        f"({df[DATE_COLUMN].min().date()} .. {df[DATE_COLUMN].max().date()})"
    )
    return df

def get_data_profile(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Generate a comprehensive data profile for EDA.

    Returns a dictionary with shape, date coverage, entity counts,
    missing values, duplicates, and target statistics.

    Args:
        df: Input DataFrame.

    Returns:
        Dictionary containing profile metrics.
    """
    n_rows, n_cols = df.shape

    missing = df.isnull().sum()
    n_duplicates = int(df.duplicated(subset=[DATE_COLUMN, "store", "item"]).sum())

    profile = {
        "n_rows": n_rows,
        "n_cols": n_cols,
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        "date_min": str(df[DATE_COLUMN].min().date()),
        "date_max": str(df[DATE_COLUMN].max().date()),
        "n_stores": int(df["store"].nunique()),
        "n_items": int(df["item"].nunique()),
        "n_series": int(df.groupby(["store", "item"]).ngroups),
        "n_missing_total": int(missing.sum()),
        "n_duplicate_keys": n_duplicates,
        "sales_mean": round(float(df[TARGET].mean()), 2),
        "sales_median": float(df[TARGET].median()),
        "sales_min": float(df[TARGET].min()),
        "sales_max": float(df[TARGET].max()),
        "n_negative_sales": int((df[TARGET] < 0).sum()),
    }

    logger.info(
        f"Profile: {n_rows:,} rows, {profile['n_series']} store-item series, "
        f"{profile['n_missing_total']} missing values, "
        f"{n_duplicates} duplicate (date, store, item) keys"
    )

    return profile

In [ ]:
download_dataset()
df_raw = load_raw_data()
profile = get_data_profile(df_raw)

print(f"Coverage: {profile['date_min']} .. {profile['date_max']}")
print(f"Series: {profile['n_series']} | Mean sales: {profile['sales_mean']}")
df_raw.head()

## 2️⃣ Cleaning

Model-agnostic fixes: parse dates (dropping unparseable rows), coerce numeric types,
remove duplicate `(date, store, item)` keys that would double-count sales, and clip
negative sales to 0.

In [ ]:
class DataCleaner(BaseEstimator, TransformerMixin):
    """
    Initial data cleaning steps applied before the main pipeline.

    Handles:
    - Parsing the date column to datetime
    - Removing duplicate (date, store, item) keys (keep first)
    - Coercing store/item/sales to numeric
    - Clipping negative sales to 0 (returns are out of scope here)
    """

    def fit(self, X: pd.DataFrame, y: Optional[pd.Series] = None) -> "DataCleaner":
        """No fitting required."""
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Clean the DataFrame."""
        df = X.copy()

        if DATE_COLUMN in df.columns:
            df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
            n_bad_dates = int(df[DATE_COLUMN].isna().sum())
            if n_bad_dates:
                logger.warning(f"Dropping {n_bad_dates} rows with unparseable dates.")
                df = df.dropna(subset=[DATE_COLUMN])

        for col in ("store", "item", TARGET):
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        # Duplicate (date, store, item) keys would double-count sales
        key_cols = [c for c in (DATE_COLUMN, "store", "item") if c in df.columns]
        if len(key_cols) == 3:
            n_before = len(df)
            df = df.drop_duplicates(subset=key_cols, keep="first")
            n_dropped = n_before - len(df)
            if n_dropped:
                logger.info(f"Dropped {n_dropped} duplicate (date, store, item) rows.")

        if TARGET in df.columns:
            n_negative = int((df[TARGET] < 0).sum())
            if n_negative:
                logger.warning(f"Clipping {n_negative} negative sales values to 0.")
                df[TARGET] = df[TARGET].clip(lower=0)

        return df

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply initial cleaning to the raw DataFrame.

    Convenience wrapper around DataCleaner for use outside sklearn
    Pipelines (e.g., in EDA notebooks).

    Args:
        df: Raw DataFrame from data_loader.

    Returns:
        Cleaned DataFrame ready for splitting / feature engineering.
    """
    cleaner = DataCleaner()
    df_clean = cleaner.transform(df)
    logger.info(
        f"Cleaning complete: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} cols"
    )
    return df_clean

In [ ]:
df_clean = clean_data(df_raw)
df_clean.head()

## 3️⃣ Exploratory Data Analysis

Ten charts saved to `figures/` — the total-sales timeline, monthly and day-of-week
seasonality, store and item comparisons, year-over-year growth, the sales
distribution, a store×month heatmap, the weekday/weekend×month interaction, and one
example series.

**Headline insights:** demand peaks every July and troughs in January (~50% swing);
weekends sell markedly more than weekdays (Sunday strongest); demand grows steadily
year over year; and stores/items keep their relative ranking — an entity's history is
a powerful predictor of its future.

In [ ]:
def run_eda(df: pd.DataFrame) -> None:
    """
    Generate all EDA visualizations and save to figures/.

    Args:
        df: Cleaned DataFrame with date, store, item, sales.
    """
    logger.info("=" * 60)
    logger.info("EXPLORATORY DATA ANALYSIS")
    logger.info("=" * 60)

    plt.style.use("seaborn-v0_8-whitegrid")
    dates = df[DATE_COLUMN]

    # 1. Total daily sales over time (trend + seasonality)
    daily = df.groupby(DATE_COLUMN)[TARGET].sum()
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(daily.index, daily.values, color="#2196F3", linewidth=0.7)
    ax.plot(daily.rolling(30, center=True).mean(), color="#FF5722",
            linewidth=2, label="30-day rolling mean")
    ax.set_title("Total Daily Sales — All Stores & Items", fontsize=13, fontweight="bold")
    ax.set_ylabel("Total sales")
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_daily_sales.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 2. Monthly seasonality
    monthly = df.groupby(dates.dt.month)[TARGET].mean()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(monthly.index, monthly.values, color="#2196F3", edgecolor="white")
    ax.set_title("Average Sales by Month — Yearly Seasonality", fontsize=13, fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_ylabel("Mean sales per store-item-day")
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_monthly_seasonality.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 3. Day-of-week pattern
    dow = df.groupby(dates.dt.dayofweek)[TARGET].mean()
    fig, ax = plt.subplots(figsize=(9, 5))
    labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    colors = ["#2196F3"] * 5 + ["#FF5722"] * 2
    ax.bar(labels, dow.values, color=colors, edgecolor="white")
    ax.set_title("Average Sales by Day of Week", fontsize=13, fontweight="bold")
    ax.set_ylabel("Mean sales per store-item-day")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_dayofweek.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 4. Sales by store
    store_sales = df.groupby("store")[TARGET].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(store_sales.index.astype(str), store_sales.values,
           color="#4CAF50", edgecolor="white")
    ax.set_title("Average Sales by Store", fontsize=13, fontweight="bold")
    ax.set_xlabel("Store")
    ax.set_ylabel("Mean sales per item-day")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_sales_by_store.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 5. Item demand spread (top & bottom 10 items)
    item_sales = df.groupby("item")[TARGET].mean().sort_values(ascending=False)
    top_bottom = pd.concat([item_sales.head(10), item_sales.tail(10)])
    fig, ax = plt.subplots(figsize=(11, 6))
    bar_colors = ["#2196F3"] * 10 + ["#FF5722"] * 10
    ax.bar(top_bottom.index.astype(str), top_bottom.values,
           color=bar_colors, edgecolor="white")
    ax.set_title("Item Demand Spread — Top 10 vs Bottom 10 Items",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Item")
    ax.set_ylabel("Mean sales per store-day")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_item_spread.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 6. Year-over-year growth
    yearly = df.groupby(dates.dt.year)[TARGET].mean()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(yearly.index, yearly.values, marker="o", color="#9C27B0", linewidth=2)
    for x, v in yearly.items():
        ax.text(x, v, f"{v:.1f}", ha="center", va="bottom", fontweight="bold")
    ax.set_title("Year-over-Year Demand Growth", fontsize=13, fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Mean sales per store-item-day")
    ax.set_xticks(yearly.index)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_yearly_growth.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 7. Sales distribution
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(df[TARGET], bins=60, color="#2196F3", edgecolor="white")
    ax.set_title("Distribution of Daily Store-Item Sales", fontsize=13, fontweight="bold")
    ax.set_xlabel("Units sold")
    ax.set_ylabel("Count")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_sales_distribution.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 8. Store × Month heatmap
    pivot = df.pivot_table(values=TARGET, index="store",
                           columns=dates.dt.month, aggfunc="mean")
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(pivot, cmap="YlOrRd", annot=True, fmt=".0f",
                linewidths=0.5, ax=ax, cbar_kws={"label": "Mean sales"})
    ax.set_title("Mean Sales — Store × Month", fontsize=13, fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_ylabel("Store")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_store_month_heatmap.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 9. Weekday vs weekend by month (interaction)
    df_tmp = df[[TARGET]].copy()
    df_tmp["month"] = dates.dt.month.values
    df_tmp["is_weekend"] = (dates.dt.dayofweek >= 5).values
    interaction = df_tmp.groupby(["month", "is_weekend"])[TARGET].mean().unstack()
    interaction.columns = ["Weekday", "Weekend"]
    fig, ax = plt.subplots(figsize=(10, 5))
    interaction.plot(kind="bar", ax=ax, color=["#2196F3", "#FF5722"], edgecolor="white")
    ax.set_title("Weekday vs Weekend Demand by Month", fontsize=13, fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_ylabel("Mean sales")
    plt.xticks(rotation=0)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_weekend_by_month.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 10. Example series — one store-item over time
    example = df[(df["store"] == 1) & (df["item"] == 1)].set_index(DATE_COLUMN)[TARGET]
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(example.index, example.values, color="#2196F3", linewidth=0.5, alpha=0.7)
    ax.plot(example.rolling(30, center=True).mean(), color="#FF5722",
            linewidth=2, label="30-day rolling mean")
    ax.set_title("Example Series — Store 1, Item 1", fontsize=13, fontweight="bold")
    ax.set_ylabel("Units sold")
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_example_series.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    logger.info(
        f"EDA complete. Saved {len(list(FIGURES_DIR.glob('eda_*.png')))} "
        f"visualizations to {FIGURES_DIR}"
    )

In [ ]:
run_eda(df_clean)

for fig_name in ["eda_daily_sales.png", "eda_monthly_seasonality.png",
                 "eda_dayofweek.png", "eda_yearly_growth.png"]:
    display(Image(filename=str(FIGURES_DIR / fig_name)))

## 4️⃣ Feature Engineering

Two families of features:

| Family | Features | Rationale |
|---|---|---|
| **Calendar** | year, month, day, dayofweek, dayofyear, weekofyear, weekend/month-start/month-end flags | The seasonal and weekly structure seen in the EDA |
| **Cyclical encodings** | sin/cos of month, day-of-week, day-of-year | December neighbours January; Sunday neighbours Monday — plain integers put them at opposite ends |
| **Learned aggregates** | store_item_mean, store_item_dow_mean, item_month_mean, store_month_mean | Each entity's demand level and rhythm — the strongest signals in this dataset |

**Leakage / train-serve-skew guard:** the `FeatureEngineer` transformer runs *inside*
the model pipeline. Its `fit()` learns every aggregate **from the training fold
only** and re-applies them identically to any future row — a single API request for
`(2020-12-25, store 10, item 50)` needs no historical data at serving time. Unseen
entities fall back through coarser aggregates to the global training mean, never NaN.

In [ ]:
def create_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive calendar and cyclical features from the date column.

    Cyclical sin/cos pairs let the models see that month 12 neighbours
    month 1 and Sunday neighbours Monday — plain integers put them at
    opposite ends of the range.
    """
    df = df.copy()
    dates = pd.to_datetime(df[DATE_COLUMN], errors="coerce")

    df["year"] = dates.dt.year
    df["month"] = dates.dt.month
    df["day"] = dates.dt.day
    df["dayofweek"] = dates.dt.dayofweek
    df["dayofyear"] = dates.dt.dayofyear
    df["weekofyear"] = dates.dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["is_month_start"] = dates.dt.is_month_start.astype(int)
    df["is_month_end"] = dates.dt.is_month_end.astype(int)

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["doy_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365.25)

    return df

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Sklearn transformer that adds calendar features and learned aggregates.

    Designed to be the FIRST step of the model pipeline:

        Pipeline([
            ("features", FeatureEngineer()),
            ("preprocessor", ColumnTransformer(...)),
            ("regressor", ...),
        ])

    `fit` learns per-entity and seasonal mean-sales statistics from the
    training fold only, so time-series cross-validation is leakage-free and
    single-row API requests get the exact same feature definitions as
    training data. Unseen (store, item) combinations fall back through
    coarser aggregates down to the global training mean.
    """

    def fit(self, X: pd.DataFrame, y=None) -> "FeatureEngineer":
        """Learn aggregate demand statistics from training data."""
        for col in (DATE_COLUMN, "store", "item"):
            if col not in X.columns:
                raise ValueError(f"FeatureEngineer requires a '{col}' column.")
        if y is None:
            raise ValueError(
                "FeatureEngineer requires y (sales) during fit to learn "
                "aggregate demand statistics."
            )

        base = X[[DATE_COLUMN, "store", "item"]].copy()
        base["_sales"] = np.asarray(y, dtype=float)
        dates = pd.to_datetime(base[DATE_COLUMN], errors="coerce")
        base["_month"] = dates.dt.month
        base["_dow"] = dates.dt.dayofweek

        self.global_mean_ = float(base["_sales"].mean())
        self.store_item_mean_ = (
            base.groupby(["store", "item"])["_sales"].mean()
            .rename("store_item_mean").reset_index()
        )
        self.store_item_dow_mean_ = (
            base.groupby(["store", "item", "_dow"])["_sales"].mean()
            .rename("store_item_dow_mean").reset_index()
        )
        self.item_month_mean_ = (
            base.groupby(["item", "_month"])["_sales"].mean()
            .rename("item_month_mean").reset_index()
        )
        self.store_month_mean_ = (
            base.groupby(["store", "_month"])["_sales"].mean()
            .rename("store_month_mean").reset_index()
        )

        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Add calendar features and learned aggregates (fit stats only)."""
        df = create_calendar_features(X)
        df["store"] = pd.to_numeric(df["store"], errors="coerce")
        df["item"] = pd.to_numeric(df["item"], errors="coerce")

        # Left-merges preserve row order; fallback cascade ends at the
        # global training mean so unseen entities never produce NaN
        df = df.merge(self.store_item_mean_, on=["store", "item"], how="left")
        df = df.merge(
            self.store_item_dow_mean_.rename(columns={"_dow": "dayofweek"}),
            on=["store", "item", "dayofweek"], how="left",
        )
        df = df.merge(
            self.item_month_mean_.rename(columns={"_month": "month"}),
            on=["item", "month"], how="left",
        )
        df = df.merge(
            self.store_month_mean_.rename(columns={"_month": "month"}),
            on=["store", "month"], how="left",
        )

        df["store_item_mean"] = df["store_item_mean"].fillna(self.global_mean_)
        df["store_item_dow_mean"] = df["store_item_dow_mean"].fillna(df["store_item_mean"])
        df["item_month_mean"] = df["item_month_mean"].fillna(self.global_mean_)
        df["store_month_mean"] = df["store_month_mean"].fillna(self.global_mean_)

        df.index = X.index
        return df

    def get_feature_names_out(self, input_features=None) -> np.ndarray:
        """Return input feature names plus the engineered feature names."""
        if input_features is None:
            input_features = self.feature_names_in_
        # Same order in which transform() appends the new columns
        engineered = [
            "year", "month", "day", "dayofweek", "dayofyear", "weekofyear",
            "is_weekend", "is_month_start", "is_month_end",
            "month_sin", "month_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos",
            "store_item_mean", "store_item_dow_mean",
            "item_month_mean", "store_month_mean",
        ]
        new = [f for f in engineered if f not in list(input_features)]
        return np.concatenate([
            np.asarray(input_features, dtype=object),
            np.asarray(new, dtype=object),
        ])

def engineer_features(
    df: pd.DataFrame,
    target: Optional[str] = "sales",
) -> pd.DataFrame:
    """
    Apply all feature engineering for EDA/notebook exploration.

    Aggregates are computed from `df` itself here — acceptable only for
    full-dataset exploration. Production code must use the FeatureEngineer
    transformer inside the model pipeline so statistics come from training
    folds only.

    Args:
        df: Cleaned DataFrame including the target column.
        target: Target column name used for the aggregates.

    Returns:
        DataFrame with all engineered features added.
    """
    fe = FeatureEngineer()
    X = df.drop(columns=[target]) if target in df.columns else df
    y = df[target] if target in df.columns else None
    fe.fit(X, y)
    out = fe.transform(X)
    if target in df.columns:
        out[target] = df[target].values
    logger.info("Feature engineering complete (exploration mode).")
    return out

In [ ]:
# Exploration-only preview (aggregates from the full data). During training
# the FeatureEngineer pipeline step recomputes everything per CV fold.
df_featured = engineer_features(df_clean)

preview_cols = ["date", "store", "item", "sales", "dayofweek", "is_weekend",
                "store_item_mean", "store_item_dow_mean", "item_month_mean"]
df_featured[preview_cols].head(8)

## 5️⃣ Chronological Train / Validation / Test Split

Strictly time-ordered — no future leaks into training:

- **Train** — 2013-01-01 .. 2017-06-30 (~90%, model fitting + time-series CV)
- **Validation** — 2017-07-01 .. 2017-09-30 (model selection)
- **Test** — 2017-10-01 .. 2017-12-31 (final unbiased estimate, matching the
  competition's 3-month horizon)

Frames are sorted by date so `TimeSeriesSplit` inside the hyperparameter search sees
chronological folds.

In [ ]:
def split_data(
    df: pd.DataFrame,
    target: str = TARGET,
    train_end: str = TRAIN_END_DATE,
    val_end: str = VAL_END_DATE,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame,
           pd.Series, pd.Series, pd.Series]:
    """
    Split data chronologically into Train / Validation / Test.

    A random split would leak future information into training (the model
    would learn from days it is later asked to predict). Instead:

        Train      : ..            .. train_end   (~90%)
        Validation : train_end+1   .. val_end     (3 months — model selection)
        Test       : val_end+1     .. end         (3 months — final estimate)

    The 3-month validation/test windows match the competition's forecast
    horizon. Output frames are sorted by date so TimeSeriesSplit inside
    the hyperparameter search sees chronological folds.

    Args:
        df: Cleaned DataFrame with date and target columns.
        target: Target column name.
        train_end: Last date (inclusive) of the training window.
        val_end: Last date (inclusive) of the validation window.

    Returns:
        Tuple of (X_train, X_val, X_test, y_train, y_val, y_test)
    """
    df = df.sort_values(DATE_COLUMN).reset_index(drop=True)

    train_mask = df[DATE_COLUMN] <= pd.Timestamp(train_end)
    val_mask = (df[DATE_COLUMN] > pd.Timestamp(train_end)) & \
               (df[DATE_COLUMN] <= pd.Timestamp(val_end))
    test_mask = df[DATE_COLUMN] > pd.Timestamp(val_end)

    feature_cols = [c for c in df.columns if c != target]
    X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, target]
    X_val, y_val = df.loc[val_mask, feature_cols], df.loc[val_mask, target]
    X_test, y_test = df.loc[test_mask, feature_cols], df.loc[test_mask, target]

    logger.info(
        f"Chronological split — "
        f"Train: {len(X_train):,} rows (.. {train_end}), "
        f"Val: {len(X_val):,} rows (.. {val_end}), "
        f"Test: {len(X_test):,} rows"
    )
    logger.info(
        f"Mean sales — Train: {y_train.mean():.2f}, "
        f"Val: {y_val.mean():.2f}, Test: {y_test.mean():.2f}"
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df_clean)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 6️⃣ Preprocessing Architecture

A `ColumnTransformer` with three explicit branches:

| Branch | Columns | Transform |
|---|---|---|
| `num` | store, item, calendar numerics, learned aggregates | median-impute (no scaling — all models are tree-based) |
| `cat` | month, dayofweek | `OneHotEncoder(drop=None, handle_unknown="ignore")` |
| `flag` | is_weekend, is_month_start, is_month_end | passthrough |

**`remainder="drop"`** is a deliberate safety net: unexpected columns in inference
data (`sales`, `id`, arbitrary CSV columns) are ignored instead of crashing or
leaking into the model.

In [ ]:
def build_preprocessing_pipeline(
    numeric_features: Optional[List[str]] = None,
    categorical_features: Optional[List[str]] = None,
    flag_features: Optional[List[str]] = None,
) -> ColumnTransformer:
    """
    Build a ColumnTransformer that handles all feature transformations.

    Architecture:
    - Numeric features (entity IDs, calendar numerics, learned aggregates)
      → median imputation. No scaling: all three models are tree-based and
      split on thresholds, so scale is irrelevant.
    - Calendar categoricals (month, dayofweek) → OneHotEncoder
      (drop=None + handle_unknown="ignore" keeps unseen values
      distinguishable from every known category)
    - Flags (is_weekend, ...) → passthrough
    - Everything else → DROPPED. Deliberate safety net: extra columns in
      inference data (`sales`, `id`, arbitrary CSV columns) are ignored
      instead of crashing or leaking into the model.

    This transformer is fit ONLY on training data to prevent leakage.

    Returns:
        Configured ColumnTransformer.
    """
    numeric_features = NUMERIC_FEATURES if numeric_features is None else numeric_features
    categorical_features = CALENDAR_CATEGORICAL if categorical_features is None else categorical_features
    flag_features = FLAG_FEATURES if flag_features is None else flag_features

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline([
        ("encoder", OneHotEncoder(
            drop=None,
            sparse_output=False,
            handle_unknown="ignore",
        )),
    ])

    transformers = [
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
    if flag_features:
        transformers.append(("flag", "passthrough", flag_features))

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )

    return preprocessor

def get_default_preprocessor() -> ColumnTransformer:
    """Build the default preprocessor using the standard feature definitions."""
    return build_preprocessing_pipeline()

In [ ]:
preprocessor = get_default_preprocessor()
preprocessor

## 7️⃣ Feature-Importance Analysis

Two complementary rankings on the fully transformed training matrix (run on random
samples — importance estimates stabilize long before 821k rows): **mutual
information** (model-free, non-linear) and **random forest importance**
(interaction-aware). Analysis only — the models train on the full feature set.

In [ ]:
def correlation_filter(
    X: pd.DataFrame,
    threshold: float = 0.95,
) -> List[Tuple[str, str, float]]:
    """
    Report feature pairs with |correlation| above the threshold.

    Cyclical encodings and their source integers are expected to correlate;
    the report documents redundancy rather than dropping columns (tree
    models are unaffected by collinearity).

    Returns:
        List of (feature_a, feature_b, correlation) tuples.
    """
    corr_matrix = X.corr().abs()
    upper_tri = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    pairs = []
    for col in upper_tri.columns:
        for row in upper_tri.index[upper_tri[col] > threshold]:
            pairs.append((row, col, float(corr_matrix.loc[row, col])))
            logger.info(
                f"High correlation: {row} ~ {col} (r={corr_matrix.loc[row, col]:.3f})"
            )

    logger.info(f"Correlation filter: {len(pairs)} pairs above |r| > {threshold}.")
    return pairs

def mutual_information_ranking(
    X: pd.DataFrame,
    y: pd.Series,
    top_n: int = 15,
    sample_n: int = 50000,
) -> pd.DataFrame:
    """
    Rank features by Mutual Information with the target (regression).

    Runs on a random sample for speed — MI estimates stabilize well below
    full-dataset size.

    Returns:
        DataFrame with features ranked by MI score.
    """
    if len(X) > sample_n:
        idx = X.sample(n=sample_n, random_state=RANDOM_STATE).index
        X_s, y_s = X.loc[idx], y.loc[idx]
    else:
        X_s, y_s = X, y

    mi_scores = mutual_info_regression(X_s, y_s, random_state=RANDOM_STATE)

    mi_df = pd.DataFrame({
        "feature": X.columns,
        "mi_score": mi_scores,
    }).sort_values("mi_score", ascending=False).reset_index(drop=True)

    logger.info(f"Top {top_n} features by Mutual Information:")
    for _, row in mi_df.head(top_n).iterrows():
        logger.info(f"  {row['feature']}: {row['mi_score']:.4f}")

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.barplot(
        data=mi_df.head(top_n), x="mi_score", y="feature",
        hue="feature", palette="viridis", legend=False, ax=ax,
    )
    ax.set_title("Feature Ranking — Mutual Information", fontsize=14, fontweight="bold")
    ax.set_xlabel("Mutual Information Score")
    ax.set_ylabel("")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "feature_selection_mi.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    return mi_df

def model_based_importance(
    X: pd.DataFrame,
    y: pd.Series,
    top_n: int = 15,
    sample_n: int = 100000,
) -> pd.DataFrame:
    """
    Rank features using Random Forest feature importances (on a sample).

    Returns:
        DataFrame with features ranked by importance.
    """
    if len(X) > sample_n:
        idx = X.sample(n=sample_n, random_state=RANDOM_STATE).index
        X_s, y_s = X.loc[idx], y.loc[idx]
    else:
        X_s, y_s = X, y

    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(X_s, y_s)

    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": rf.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    logger.info(f"Top {top_n} features by Random Forest importance:")
    for _, row in importance_df.head(top_n).iterrows():
        logger.info(f"  {row['feature']}: {row['importance']:.4f}")

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.barplot(
        data=importance_df.head(top_n), x="importance", y="feature",
        hue="feature", palette="magma", legend=False, ax=ax,
    )
    ax.set_title("Feature Ranking — Random Forest Importance", fontsize=14, fontweight="bold")
    ax.set_xlabel("Feature Importance")
    ax.set_ylabel("")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "feature_selection_rf.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    return importance_df

def select_features(
    X: pd.DataFrame,
    y: pd.Series,
    top_n: int = 15,
    corr_threshold: float = 0.95,
) -> Tuple[List[str], Dict[str, pd.DataFrame]]:
    """
    Run the full feature-importance analysis.

    Returns:
        Tuple of (union of top features from MI + RF, dict of ranking DataFrames).
    """
    logger.info("=" * 60)
    logger.info("FEATURE-IMPORTANCE ANALYSIS")
    logger.info("=" * 60)

    correlation_filter(X, threshold=corr_threshold)
    mi_df = mutual_information_ranking(X, y, top_n=top_n)
    rf_df = model_based_importance(X, y, top_n=top_n)

    mi_top = set(mi_df.head(top_n)["feature"].tolist())
    rf_top = set(rf_df.head(top_n)["feature"].tolist())
    selected = sorted(mi_top | rf_top)

    logger.info(f"Strong features (union of MI + RF top-{top_n}): {len(selected)}")
    for feat in selected:
        logger.info(f"  ✓ {feat}")

    rankings = {"mutual_information": mi_df, "random_forest": rf_df}
    return selected, rankings

In [ ]:
# Transform the training data exactly the way the models will see it
analysis_pipeline = Pipeline([
    ("features", FeatureEngineer()),          # aggregates learned from X_train only
    ("preprocessor", get_default_preprocessor()),
])
X_train_transformed = analysis_pipeline.fit_transform(X_train, y_train)
feature_names_out = list(analysis_pipeline.named_steps["preprocessor"].get_feature_names_out())
X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names_out)

selected_features, rankings = select_features(X_train_df, y_train.reset_index(drop=True), top_n=15)

display(Image(filename=str(FIGURES_DIR / "feature_selection_mi.png")))
display(Image(filename=str(FIGURES_DIR / "feature_selection_rf.png")))

del X_train_transformed, X_train_df   # free ~1GB before training

## 8️⃣ Model Training & Hyperparameter Tuning

Three candidates — the histogram-based gradient-boosting family that dominates this
competition's leaderboard — each as a self-contained pipeline
`FeatureEngineer → ColumnTransformer → regressor`:

- **LightGBM** — leaf-wise growth, typically the strongest here.
- **XGBoost** — depth-wise with strong regularization.
- **HistGradientBoosting** — scikit-learn's native LightGBM-style GBM.

Each runs `RandomizedSearchCV` optimizing **SMAPE** (the competition metric) with
`TimeSeriesSplit`: every CV fold validates on data strictly *after* its training
window, mirroring real forecasting. Because feature engineering sits inside the
pipeline, every fold re-learns its aggregates on its own window — no leakage.

> ⚙️ The regressors run single-threaded (`n_jobs=1`) while the search parallelizes
> across processes — nested thread pools oversubscribe cores and can crash loky
> workers on Windows. This is the long cell: **expect ~15 minutes**.

In [ ]:
# SMAPE — the competition metric (defined before the scorer that wraps it)
def smape(y_true, y_pred) -> float:
    """
    Symmetric Mean Absolute Percentage Error (the competition metric).

        SMAPE = 100/n * sum( 2*|F - A| / (|A| + |F|) )

    Rows where both actual and forecast are 0 contribute 0 (not NaN).
    Lower is better; range [0, 200].
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = np.abs(y_true) + np.abs(y_pred)
    diff = np.where(denominator == 0, 0.0, 2.0 * np.abs(y_pred - y_true) / np.where(denominator == 0, 1.0, denominator))
    return float(100.0 * np.mean(diff))


# Scorer for RandomizedSearchCV (sklearn maximizes -> negate)
SMAPE_SCORER = make_scorer(smape, greater_is_better=False)


def get_model_configs() -> Dict[str, Dict[str, Any]]:
    """
    Define all model configurations with their regressors and param grids.

    Returns:
        Dictionary mapping model names to config dicts containing:
        - 'regressor': sklearn-compatible regressor instance
        - 'params': hyperparameter search space
        - 'description': human-readable model description
    """
    configs = {
        "LightGBM": {
            "regressor": lgb.LGBMRegressor(
                random_state=RANDOM_STATE,
                verbosity=-1,
                n_jobs=1,  # search parallelizes across processes; nested
                           # thread pools can crash loky workers on Windows
            ),
            "params": LIGHTGBM_PARAMS,
            "description": (
                "Leaf-wise gradient boosting. Histogram-based splitting for "
                "speed; typically the strongest model on this competition."
            ),
        },
        "XGBoost": {
            "regressor": xgb.XGBRegressor(
                random_state=RANDOM_STATE,
                tree_method="hist",
                verbosity=0,
                n_jobs=1,  # see LightGBM note
            ),
            "params": XGBOOST_PARAMS,
            "description": (
                "Depth-wise gradient boosting with L1/L2 regularization "
                "and histogram tree construction."
            ),
        },
        "HistGradientBoosting": {
            "regressor": HistGradientBoostingRegressor(
                random_state=RANDOM_STATE,
            ),
            "params": HISTGB_PARAMS,
            "description": (
                "scikit-learn's native histogram gradient boosting "
                "(LightGBM-inspired), dependency-free and fast."
            ),
        },
    }

    return configs

def build_model_pipeline(
    preprocessor: ColumnTransformer,
    regressor: Any,
) -> Pipeline:
    """
    Create an sklearn Pipeline: feature engineering → preprocessing → regressor.

    The pipeline ensures that:
    - Learned aggregates and all preprocessing are fit ONLY on training data
    - Time-series cross-validation properly re-fits every step per fold
    - The serialized pipeline is fully self-contained for deployment:
      it accepts raw (date, store, item) records

    Args:
        preprocessor: Unfitted ColumnTransformer (cloned per model).
        regressor: sklearn-compatible regressor instance.

    Returns:
        sklearn Pipeline.
    """
    return Pipeline([
        ("features", FeatureEngineer()),
        ("preprocessor", clone(preprocessor)),
        ("regressor", regressor),
    ])

def train_single_model(
    pipeline: Pipeline,
    param_grid: Dict[str, Any],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cv_folds: int = CV_FOLDS,
    n_iter: int = N_ITER_SEARCH,
) -> Tuple[Pipeline, Dict[str, Any], float]:
    """
    Train a single model with hyperparameter tuning via RandomizedSearchCV.

    Uses TimeSeriesSplit so every CV fold validates on data strictly after
    its training window (X_train must be sorted by date — split_data
    guarantees this). Optimizes SMAPE, the competition metric.

    Returns:
        Tuple of (best pipeline, best params, training time in seconds).
    """
    cv = TimeSeriesSplit(n_splits=cv_folds)

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grid,
        n_iter=min(n_iter, _count_param_combinations(param_grid)),
        cv=cv,
        scoring=SMAPE_SCORER,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
        return_train_score=False,
    )

    start_time = time.time()
    search.fit(X_train, y_train)
    train_time = time.time() - start_time

    best_params = search.best_params_
    best_score = -search.best_score_  # scorer is negated (lower SMAPE = better)

    logger.info(
        f"Best CV SMAPE: {best_score:.3f} (training time: {train_time:.1f}s)"
    )
    logger.info(f"Best params: {best_params}")

    return search.best_estimator_, best_params, train_time

def _count_param_combinations(param_grid: Dict[str, Any]) -> int:
    """Count the total number of parameter combinations in a grid."""
    total = 1
    for values in param_grid.values():
        if isinstance(values, list):
            total *= len(values)
    return total

def train_all_models(
    preprocessor: ColumnTransformer,
    X_train: pd.DataFrame,
    y_train: pd.Series,
) -> Dict[str, Dict[str, Any]]:
    """
    Train all models with hyperparameter tuning.

    Args:
        preprocessor: ColumnTransformer for feature transformation.
        X_train: Training features (date-sorted).
        y_train: Training target (sales).

    Returns:
        Dictionary mapping model names to result dicts containing:
        - 'pipeline': Best fitted pipeline
        - 'best_params': Best hyperparameters
        - 'train_time': Training time in seconds
        - 'description': Model description
    """
    model_configs = get_model_configs()
    results = {}

    for name, config in model_configs.items():
        logger.info("=" * 60)
        logger.info(f"Training: {name}")
        logger.info("=" * 60)

        pipeline = build_model_pipeline(
            preprocessor=preprocessor,
            regressor=config["regressor"],
        )

        best_pipeline, best_params, train_time = train_single_model(
            pipeline=pipeline,
            param_grid=config["params"],
            X_train=X_train,
            y_train=y_train,
        )

        results[name] = {
            "pipeline": best_pipeline,
            "best_params": best_params,
            "train_time": train_time,
            "description": config["description"],
        }

    logger.info("=" * 60)
    logger.info("All models trained successfully.")
    logger.info("=" * 60)

    return results

In [ ]:
model_results = train_all_models(preprocessor, X_train, y_train)

## 9️⃣ Evaluation, Comparison & Model Selection

SMAPE, RMSE, MAE and R² on **all three chronological splits**, predicted-vs-actual
and residual diagnostics per model, and an aggregate daily forecast-vs-actual
timeline over the test window. Predictions are clipped at 0 — demand can't be
negative.

**The winner is chosen by *validation* SMAPE** — selecting on the test window would
leak test information into model choice. All three models typically land within
~0.1 SMAPE of each other; the champion's test SMAPE (~12.3) is competitive with
leading public solutions (~12.5–14).

*A note on the negative overfit gap:* validation SMAPE comes out *lower* than train
SMAPE because the validation window (July–September) sits on the summer demand peak —
higher sales volumes mean smaller relative errors, not an underfit model.

In [ ]:
def compute_metrics(
    y_true: pd.Series,
    y_pred: np.ndarray,
    dataset_name: str = "Test",
) -> Dict[str, float]:
    """
    Compute all regression metrics for a single dataset split.

    Args:
        y_true: Ground truth sales.
        y_pred: Predicted sales.
        dataset_name: Name of the dataset split (for logging).

    Returns:
        Dictionary of metric name → value.
    """
    metrics = {
        "smape": smape(y_true, y_pred),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

    logger.info(
        f"{dataset_name} metrics — "
        f"SMAPE: {metrics['smape']:.3f}, "
        f"RMSE: {metrics['rmse']:.3f}, "
        f"MAE: {metrics['mae']:.3f}, "
        f"R²: {metrics['r2']:.4f}"
    )

    return metrics

def evaluate_model(
    pipeline: Pipeline,
    X_train: pd.DataFrame, y_train: pd.Series,
    X_val: pd.DataFrame, y_val: pd.Series,
    X_test: pd.DataFrame, y_test: pd.Series,
    model_name: str,
) -> Dict[str, Dict[str, float]]:
    """
    Evaluate a single model on all three chronological splits.

    Returns:
        Nested dict: {split_name: {metric_name: value}}.
    """
    logger.info(f"\n--- Evaluating: {model_name} ---")
    results = {}

    for split_name, X, y in [
        ("train", X_train, y_train),
        ("validation", X_val, y_val),
        ("test", X_test, y_test),
    ]:
        y_pred = np.clip(pipeline.predict(X), 0, None)  # demand can't be negative
        results[split_name] = compute_metrics(y, y_pred, split_name.title())

    # Overfitting check: validation vs train SMAPE gap
    gap = results["validation"]["smape"] - results["train"]["smape"]
    if gap > 3.0:
        logger.warning(
            f"⚠ Possible overfitting: Val SMAPE ({results['validation']['smape']:.3f}) - "
            f"Train SMAPE ({results['train']['smape']:.3f}) = {gap:.3f}"
        )
    else:
        logger.info(f"✓ No significant overfitting: Val-Train SMAPE gap = {gap:.3f}")

    return results

def plot_predicted_vs_actual(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    model_name: str,
    dataset_name: str = "Test",
    sample_n: int = 5000,
) -> None:
    """Scatter of predicted vs actual sales on a sample of the split."""
    if len(X) > sample_n:
        idx = X.sample(n=sample_n, random_state=42).index
        X_s, y_s = X.loc[idx], y.loc[idx]
    else:
        X_s, y_s = X, y
    y_pred = np.clip(pipeline.predict(X_s), 0, None)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(y_s, y_pred, s=6, alpha=0.25, color="#2196F3", edgecolors="none")
    lims = [0, max(float(y_s.max()), float(y_pred.max())) * 1.05]
    ax.plot(lims, lims, "k--", alpha=0.6, label="Perfect forecast")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Actual sales", fontsize=12)
    ax.set_ylabel("Predicted sales", fontsize=12)
    ax.set_title(f"Predicted vs Actual — {model_name} ({dataset_name})",
                 fontsize=13, fontweight="bold")
    ax.legend()
    plt.tight_layout()
    filename = f"pred_vs_actual_{model_name.lower().replace(' ', '_')}.png"
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)

def plot_residuals(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    model_name: str,
    sample_n: int = 20000,
) -> None:
    """Residual distribution on the test split."""
    if len(X) > sample_n:
        idx = X.sample(n=sample_n, random_state=42).index
        X_s, y_s = X.loc[idx], y.loc[idx]
    else:
        X_s, y_s = X, y
    residuals = y_s - np.clip(pipeline.predict(X_s), 0, None)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(residuals, bins=60, color="#FF5722", alpha=0.75, edgecolor="white")
    ax.axvline(0, color="black", linestyle="--", alpha=0.6)
    ax.set_xlabel("Residual (actual − predicted)", fontsize=12)
    ax.set_ylabel("Count", fontsize=12)
    ax.set_title(f"Residual Distribution — {model_name} (Test)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    filename = f"residuals_{model_name.lower().replace(' ', '_')}.png"
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)

def plot_forecast_vs_actual_timeline(
    model_results: Dict[str, Dict[str, Any]],
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> None:
    """
    Aggregate daily forecast vs actual over the test window, all models.

    Summing across all 500 store-item series gives a readable one-line
    view of how well each model tracks level, trend, and weekly rhythm.
    """
    fig, ax = plt.subplots(figsize=(13, 6))
    dates = pd.to_datetime(X_test[DATE_COLUMN])

    actual_daily = y_test.groupby(dates.values).sum()
    ax.plot(actual_daily.index, actual_daily.values, color="black",
            linewidth=2, label="Actual")

    colors = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0"]
    for idx, (name, result) in enumerate(model_results.items()):
        y_pred = np.clip(result["pipeline"].predict(X_test), 0, None)
        pred_daily = pd.Series(y_pred).groupby(dates.values).sum()
        ax.plot(pred_daily.index, pred_daily.values,
                color=colors[idx % len(colors)], linewidth=1.4,
                alpha=0.85, label=f"{name} forecast")

    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Total daily sales (all stores & items)", fontsize=12)
    ax.set_title("Test Window — Aggregate Forecast vs Actual",
                 fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "forecast_vs_actual_test.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Saved aggregate forecast-vs-actual plot.")

def plot_comparison_bar_chart(comparison_df: pd.DataFrame) -> None:
    """Grouped bar chart comparing error metrics across models."""
    metrics = ["Test SMAPE", "Test RMSE", "Test MAE"]
    models = comparison_df["Model"].tolist()

    fig, axes = plt.subplots(1, len(metrics), figsize=(14, 5))
    colors = ["#2196F3", "#FF5722", "#4CAF50"]
    for ax, metric in zip(axes, metrics):
        values = comparison_df[metric].values
        ax.bar(models, values, color=colors[:len(models)], alpha=0.85, edgecolor="white")
        ax.set_title(metric, fontsize=12, fontweight="bold")
        ax.tick_params(axis="x", rotation=20)
        for i, v in enumerate(values):
            ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
    plt.suptitle("Model Performance Comparison (lower is better)",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "model_comparison_bar.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Saved model comparison bar chart.")

def create_comparison_table(
    all_metrics: Dict[str, Dict[str, Dict[str, float]]],
    training_times: Dict[str, float],
    descriptions: Dict[str, str],
) -> pd.DataFrame:
    """
    Create a comprehensive model comparison table.

    Returns:
        DataFrame with one row per model, sorted by validation SMAPE
        ascending (the selection metric; test columns are reported but
        never used to pick the winner).
    """
    rows = []
    for name in all_metrics:
        test = all_metrics[name].get("test", {})
        val = all_metrics[name].get("validation", {})
        train = all_metrics[name].get("train", {})

        rows.append({
            "Model": name,
            "Test SMAPE": test.get("smape", np.inf),
            "Test RMSE": test.get("rmse", np.inf),
            "Test MAE": test.get("mae", np.inf),
            "Test R2": test.get("r2", 0),
            "Val SMAPE": val.get("smape", np.inf),
            "Train SMAPE": train.get("smape", np.inf),
            "Overfit Gap": val.get("smape", np.inf) - train.get("smape", np.inf),
            "Training Time (s)": training_times.get(name, 0),
            "Description": descriptions.get(name, ""),
        })

    comparison_df = pd.DataFrame(rows)
    # Rank by VALIDATION SMAPE: choosing the winner on the test set would
    # leak test information into model selection and bias the reported score.
    comparison_df = comparison_df.sort_values("Val SMAPE", ascending=True)
    comparison_df = comparison_df.reset_index(drop=True)

    comparison_df.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
    logger.info(f"Model comparison table saved to {REPORTS_DIR / 'model_comparison.csv'}")

    display_cols = [
        "Model", "Test SMAPE", "Test RMSE", "Test MAE", "Test R2",
        "Val SMAPE", "Overfit Gap", "Training Time (s)",
    ]
    logger.info("\n" + comparison_df[display_cols].to_string(index=False))

    return comparison_df

def evaluate_all_models(
    model_results: Dict[str, Dict[str, Any]],
    X_train: pd.DataFrame, y_train: pd.Series,
    X_val: pd.DataFrame, y_val: pd.Series,
    X_test: pd.DataFrame, y_test: pd.Series,
) -> pd.DataFrame:
    """
    Run the complete evaluation pipeline for all trained models.

    Steps:
    1. Compute metrics on all chronological splits for each model
    2. Predicted-vs-actual and residual diagnostics per model
    3. Aggregate forecast-vs-actual timeline over the test window
    4. Comparison table (ranked by validation SMAPE) and bar chart

    Returns:
        Comparison DataFrame.
    """
    logger.info("=" * 60)
    logger.info("MODEL EVALUATION")
    logger.info("=" * 60)

    all_metrics = {}
    training_times = {}
    descriptions = {}

    for name, result in model_results.items():
        pipeline = result["pipeline"]

        metrics = evaluate_model(
            pipeline,
            X_train, y_train,
            X_val, y_val,
            X_test, y_test,
            model_name=name,
        )
        all_metrics[name] = metrics
        training_times[name] = result["train_time"]
        descriptions[name] = result["description"]

        plot_predicted_vs_actual(pipeline, X_test, y_test, name, "Test")
        plot_residuals(pipeline, X_test, y_test, name)

    plot_forecast_vs_actual_timeline(model_results, X_test, y_test)

    comparison_df = create_comparison_table(all_metrics, training_times, descriptions)
    plot_comparison_bar_chart(comparison_df)

    best_model_name = comparison_df.iloc[0]["Model"]
    logger.info(f"\n🏆 Best model: {best_model_name} (lowest Validation SMAPE)")

    return comparison_df

In [ ]:
comparison_df = evaluate_all_models(
    model_results, X_train, y_train, X_val, y_val, X_test, y_test,
)

best_model_name = comparison_df.iloc[0]["Model"]   # ranked by Val SMAPE
best_pipeline = model_results[best_model_name]["pipeline"]
print(f"Best model (by validation SMAPE): {best_model_name}")

comparison_df.drop(columns=["Description"])

In [ ]:
display(Image(filename=str(FIGURES_DIR / "forecast_vs_actual_test.png")))
display(Image(filename=str(FIGURES_DIR / "model_comparison_bar.png")))

In [ ]:
# Final summary — matches models/model_metadata.json written by main.py
best_row = comparison_df.iloc[0]
model_summary = {
    "best_model": best_model_name,
    "val_smape": round(float(best_row["Val SMAPE"]), 4),
    "test_smape": round(float(best_row["Test SMAPE"]), 4),
    "test_rmse": round(float(best_row["Test RMSE"]), 4),
    "test_mae": round(float(best_row["Test MAE"]), 4),
    "test_r2": round(float(best_row["Test R2"]), 4),
}
model_summary

## 🔟 SHAP Explainability

Game-theoretic feature attributions with the fast, exact `TreeExplainer` (all three
candidates are tree ensembles). Outputs: beeswarm summary, global importance bar
chart, single-forecast waterfall, and dependence plots for the top-3 features.

Consistent top drivers: the learned **store-item day-of-week average** (who sells
what, on which weekday), the **seasonal item-month average**, and **year** (the
growth trend) — exactly the structure the EDA revealed.

In [ ]:
def transform_features(pipeline: Pipeline, X: pd.DataFrame) -> pd.DataFrame:
    """
    Run raw features through every pipeline step except the final regressor
    and return the result as a DataFrame with proper feature names.

    Args:
        pipeline: Fitted Pipeline (feature engineering + preprocessing + regressor).
        X: Raw input features (date, store, item).

    Returns:
        Transformed DataFrame in model-input space.
    """
    transformer = Pipeline(pipeline.steps[:-1])
    X_transformed = transformer.transform(X)
    if hasattr(X_transformed, "toarray"):
        X_transformed = X_transformed.toarray()

    preprocessor = pipeline.named_steps["preprocessor"]
    try:
        feature_names = list(preprocessor.get_feature_names_out())
    except Exception:
        logger.warning("Could not extract feature names from preprocessor.")
        feature_names = [f"feature_{i}" for i in range(X_transformed.shape[1])]

    return pd.DataFrame(np.asarray(X_transformed), columns=feature_names)

def get_shap_explainer(
    pipeline: Pipeline,
    model_name: str = "Model",
) -> shap.Explainer:
    """
    Create a SHAP TreeExplainer for the pipeline's regressor.

    All candidate models (LightGBM, XGBoost, HistGradientBoosting) are tree
    ensembles supported by the fast, exact TreeExplainer.
    """
    regressor = pipeline.named_steps["regressor"]
    logger.info(f"Using TreeExplainer for {model_name} ({type(regressor).__name__})")
    return shap.TreeExplainer(regressor)

def compute_shap_values(
    pipeline: Pipeline,
    X: pd.DataFrame,
    explainer: shap.Explainer,
    max_samples: int = 2000,
) -> tuple:
    """
    Compute SHAP values for a (sampled) dataset.

    Args:
        pipeline: Fitted Pipeline.
        X: Raw features (before preprocessing).
        explainer: SHAP Explainer instance.
        max_samples: Maximum number of rows to explain (for speed).

    Returns:
        Tuple of (shap_values, X_transformed as DataFrame).
    """
    if len(X) > max_samples:
        X_sample = X.sample(n=max_samples, random_state=42)
    else:
        X_sample = X

    X_df = transform_features(pipeline, X_sample)

    logger.info(f"Computing SHAP values for {len(X_df)} samples...")
    shap_values = explainer.shap_values(X_df)

    # Regression explainers return a 2D array; guard list/3D just in case
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    elif getattr(shap_values, "ndim", 2) == 3:
        shap_values = shap_values[:, :, 0]

    return shap_values, X_df

def plot_shap_summary(shap_values, X_df, model_name="Model") -> None:
    """SHAP beeswarm summary — direction and magnitude of every feature."""
    fig = plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_df, plot_type="dot", show=False, max_display=15)
    plt.title(f"SHAP Summary — {model_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "shap_summary.png", dpi=150, bbox_inches="tight")
    plt.close("all")
    logger.info("Saved SHAP summary plot (beeswarm).")

def plot_shap_bar(shap_values, X_df, model_name="Model") -> None:
    """SHAP bar plot — global mean |SHAP| importance."""
    fig = plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_df, plot_type="bar", show=False, max_display=15)
    plt.title(f"Global Feature Importance (SHAP) — {model_name}",
              fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "shap_bar.png", dpi=150, bbox_inches="tight")
    plt.close("all")
    logger.info("Saved SHAP bar plot (global importance).")

def plot_shap_waterfall(
    explainer, shap_values, X_df, sample_idx: int = 0, model_name: str = "Model",
) -> None:
    """SHAP waterfall for a single forecast — the best stakeholder view."""
    try:
        expected = explainer.expected_value
        if isinstance(expected, (list, np.ndarray)):
            expected = np.atleast_1d(expected)[0]

        explanation = shap.Explanation(
            values=shap_values[sample_idx],
            base_values=expected,
            data=X_df.iloc[sample_idx].values,
            feature_names=X_df.columns.tolist(),
        )

        fig = plt.figure(figsize=(10, 8))
        shap.plots.waterfall(explanation, show=False, max_display=12)
        plt.title(f"SHAP Waterfall — Single Forecast ({model_name})",
                  fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "shap_waterfall.png", dpi=150, bbox_inches="tight")
        plt.close("all")
        logger.info("Saved SHAP waterfall plot.")
    except Exception as e:
        logger.warning(f"Could not create waterfall plot: {e}")

def plot_shap_dependence(
    shap_values, X_df, top_n: int = 3, model_name: str = "Model",
) -> None:
    """Dependence plots for the top-N most important features."""
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top_indices = np.argsort(mean_abs_shap)[-top_n:][::-1]
    top_features = [X_df.columns[i] for i in top_indices]

    for feature in top_features:
        try:
            fig, ax = plt.subplots(figsize=(8, 5))
            shap.dependence_plot(feature, shap_values, X_df, show=False, ax=ax)
            ax.set_title(f"SHAP Dependence — {feature} ({model_name})",
                         fontsize=13, fontweight="bold")
            plt.tight_layout()
            safe_name = feature.replace(" ", "_").replace("/", "_")
            fig.savefig(FIGURES_DIR / f"shap_dependence_{safe_name}.png",
                        dpi=150, bbox_inches="tight")
            plt.close(fig)
        except Exception as e:
            logger.warning(f"Could not create dependence plot for {feature}: {e}")

    logger.info(f"Saved SHAP dependence plots for top {top_n} features.")

def run_explainability(
    pipeline: Pipeline,
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    model_name: str = "Best Model",
) -> Dict[str, Any]:
    """
    Run the full SHAP explainability pipeline.

    Args:
        pipeline: Fitted best-model Pipeline.
        X_train: Training features (unused for tree explainers; kept for parity).
        X_test: Test features (to explain).
        model_name: Model name for plot titles.

    Returns:
        Dictionary with SHAP values and feature importance ranking.
    """
    logger.info("=" * 60)
    logger.info(f"SHAP EXPLAINABILITY — {model_name}")
    logger.info("=" * 60)

    explainer = get_shap_explainer(pipeline, model_name)
    shap_values, X_df = compute_shap_values(pipeline, X_test, explainer)

    plot_shap_summary(shap_values, X_df, model_name)
    plot_shap_bar(shap_values, X_df, model_name)
    plot_shap_waterfall(explainer, shap_values, X_df, sample_idx=0, model_name=model_name)
    plot_shap_dependence(shap_values, X_df, top_n=3, model_name=model_name)

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    importance_df = pd.DataFrame({
        "feature": X_df.columns,
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    logger.info("\nTop 10 features by SHAP importance:")
    for _, row in importance_df.head(10).iterrows():
        logger.info(f"  {row['feature']}: {row['mean_abs_shap']:.4f}")

    return {
        "shap_values": shap_values,
        "feature_importance": importance_df,
        "explainer": explainer,
    }

In [ ]:
shap_results = run_explainability(
    best_pipeline, X_train, X_test, model_name=best_model_name,
)

display(Image(filename=str(FIGURES_DIR / "shap_summary.png")))
display(Image(filename=str(FIGURES_DIR / "shap_bar.png")))
display(Image(filename=str(FIGURES_DIR / "shap_waterfall.png")))

## ✅ Wrap-Up

This notebook reproduced the full data-science pipeline — cleaning, EDA, feature
engineering, chronological splitting, training, validation, comparison, and
explainability — using code identical to the production scripts, and regenerated the
same figures and `reports/model_comparison.csv`.

**Two things intentionally stay in `python main.py`:**
- **Serving-artifact persistence** (`models/final_pipeline.joblib` & metadata) —
  pipelines pickled from a notebook reference `__main__`-defined classes, which the
  FastAPI service could not unpickle. The deployable artifact must come from the
  scripts.
- **MLflow experiment tracking** (`mlruns/`) and the processed-split CSV exports.

Serving stack: `uvicorn app.api:app` (REST API with strict validation and automatic
model reload) and `streamlit run app/streamlit_app.py` (dashboard with 90-day
forecast charts and SHAP waterfalls).